# Pré-processamento e CNN

## Importações

In [ ]:
import json
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from torch.utils.data import DataLoader

from nucleo import (
    transformacao, dividir_arquivos, DatasetReconstrucao, AutoencoderCNN,
    calibrar_lambda, treinar,
    SEED, SEED_VALIDACAO, TEMPERATURA_PADRAO
)


## Configurações

In [ ]:
PASTA_TILE = Path("datasets/MVTecAD/tile")
PASTA_MODELOS = Path("modelos")

# Otimizados
BATCH_SIZE = 16
EPOCAS = 30
TAXA_APRENDIZADO = 1e-3

TEMPERATURA = TEMPERATURA_PADRAO     # suavizacao da caracteristica de Euler

# Peso do termo de Euler. Se LAMBDA_EULER = None, ele é calibrado para que o termo de Euler valha PESO_EULER_RELATIVO x o MSE na escala inicial dos erros.
PESO_EULER_RELATIVO = 0.1
LAMBDA_EULER = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Pré-processamento

In [ ]:
# Divisão do dataset. OBS.:
# Treinamento : dano NOVO a cada acesso (aumento de dados)
# Validacao   : danos FIXOS (curva de validacao comparavel entre epocas)

arquivos_treino, arquivos_validacao = dividir_arquivos(PASTA_TILE / "train" / "good", seed=SEED)

dataset_treino = DatasetReconstrucao(arquivos_treino, transformacao)
dataset_validacao = DatasetReconstrucao(arquivos_validacao, transformacao, danos_fixos=True, semente=SEED_VALIDACAO)

loader_treino = DataLoader(dataset_treino, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
loader_validacao = DataLoader(dataset_validacao, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Treinamento:", len(dataset_treino))
print("Validacao:", len(dataset_validacao))

# Visualização de pares

entrada, alvo = next(iter(loader_treino))
fig, eixos = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    eixos[0, i].imshow(entrada[i, 0], cmap="gray", vmin=0, vmax=1)
    eixos[0, i].set_title("Danificada")
    eixos[1, i].imshow(alvo[i, 0], cmap="gray", vmin=0, vmax=1)
    eixos[1, i].set_title("Original")
    eixos[0, i].axis("off")
    eixos[1, i].axis("off")

plt.tight_layout()
plt.show()


## Calibração do peso do termo de euler

In [ ]:
if LAMBDA_EULER is None:
    lambda_euler, mse_ref, euler_ref = calibrar_lambda(loader_treino, PESO_EULER_RELATIVO, temperatura=TEMPERATURA)
    print(f"MSE (danificada vs original)      : {mse_ref:.5f}")
    print(f"|dChi| (danificada vs original)   : {euler_ref:.2f}")

else:
    lambda_euler = LAMBDA_EULER

print(f"lambda_euler = {lambda_euler:.6f}")


## Rede - Autoencoder convolucional com característica de euler
perda = MSE + lambda * media(|chi(reconstruida) - chi(original)|)

In [ ]:
torch.manual_seed(SEED)

modelo = AutoencoderCNN().to(dispositivo)

print("Parametros:", sum(p.numel() for p in modelo.parameters()))

historico = treinar(
    modelo,
    loader_treino,
    loader_validacao,
    lambda_euler=lambda_euler,
    epocas=EPOCAS,
    taxa_aprendizado=TAXA_APRENDIZADO,
    dispositivo=dispositivo,
    caminho_modelo=PASTA_MODELOS / "autoencoder_euler.pth",
    temperatura=TEMPERATURA
)


In [ ]:
# Visualização das curvas de treinamento

fig, eixos = plt.subplots(1, 3, figsize=(16, 4.5))

eixos[0].plot(historico["treino"], label="Treinamento")
eixos[0].plot(historico["validacao"], label="Validacao")
eixos[0].set_title("Perda total")

eixos[1].plot(historico["mse_validacao"], color="tab:green")
eixos[1].set_title("MSE (validacao)")

eixos[2].plot(historico["euler_validacao"], color="tab:red")
eixos[2].set_title("|dChi| medio (validacao)")

for eixo in eixos:
    eixo.set_xlabel("Epoca")
    eixo.grid(alpha=0.3)

eixos[0].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Baseline - modelo sem euler

torch.manual_seed(SEED)
modelo_base = AutoencoderCNN().to(dispositivo)
random.seed(SEED)

historico_base = treinar(
    modelo_base,
    loader_treino,
    loader_validacao,
    lambda_euler=0.0,
    epocas=EPOCAS,
    taxa_aprendizado=TAXA_APRENDIZADO,
    dispositivo=dispositivo,
    caminho_modelo=PASTA_MODELOS / "autoencoder_baseline.pth",
    temperatura=TEMPERATURA
)
